# WoodelfHD — Depth Sweep (Notebook 1 / 3)

Runs **WoodelfHD** (`woodelf_for_high_depth`) across all datasets and task types
in the `woodelfhd_depth_sweep_experiment`.  
Notebooks 02 and 03 run OriginalWoodelf and SHAP **in parallel** on separate sessions.
Notebook 04 merges all three result files and generates the HTML report.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` and `woodelf_explainer` repos
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method woodelf_hd`
5. Writes partial results to Drive as `partial_woodelf_hd.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | UCI direct download (~8 GB uncompressed — takes ~20 min) |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Edit DRIVE_FOLDER to point to your preferred Google Drive folder.
# All other paths are derived from it.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

# The framework writes the method cache file directly to this path.
# Notebook 04 copies it to the local cache to generate the final report.
DRIVE_RESULT_PATH = DRIVE_FOLDER / 'woodelf_hd.json'
print(f'Method cache will be saved to: {DRIVE_RESULT_PATH}')

In [ ]:
# ── Step 3: Clone repositories ───────────────────────────────────────────────
# Replace the two URLs below with the actual repository URLs.

TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'  # e.g. https://github.com/you/treebranchmarks.git
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'          # e.g. https://github.com/you/woodelf_explainer.git

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# woodelf_explainer must be installed before treebranchmarks (it is listed
# as a dependency in treebranchmarks/pyproject.toml).

!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
# The framework writes the method cache file to Drive after every approach result.
# On restart, copy it back to the local cache directory so the experiment
# skips already-completed entries and only runs what is still missing.
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'woodelf_hd.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

In [ ]:
# ── Step 6: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write method cache to Drive after every approach result

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd \
    --result_location "{DRIVE_RESULT_PATH}"

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')